# Universal Imports

In [2]:
import pandas as pd
from sqlalchemy import text
import numpy as np
import xml.etree.ElementTree as ET

import os
import sys
import time
import datetime as dt
from pathlib import Path

In [4]:
# Importing the sqlite engine from custom tool/function file
from core.functions import engine

# Grabbing today's date
today = dt.datetime.now().strftime('%Y-%m-%d') # Format: '2025-12-10' December 10th, 2025  datetime.now

# Printing today's date for logging step
print(f"Today's date: {today}")

Today's date: 2026-09-13


# Converting raw health xml data to csv

In [ ]:
# Defining all the functions to preprocess the apple health xml file into more useable csv file that'll be saved to apple directory

def strip_invisible_character(line):
    return line.replace("\x0b", "")

def preprocess_to_temp_file(file_path):
    """
    The export.xml file is where all your data is, but Apple Health Export has
    two main problems that make it difficult to parse: 
        1. The DTD markup syntax is exported incorrectly by Apple Health for some data types.
        2. The invisible character \x0b (sometimes rendered as U+000b) likes to destroy trees. Think of the trees!

    Knowing this, we can save the trees and pre-processes the XML data to avoid destruction and ParseErrors.
    """

    print("Pre-processing and writing to temporary file...", end="")
    sys.stdout.flush()

    # Saving the preprocessed xml file in same directory (not important as it will  be deleted)
    temp_file_path = r"temp_preprocessed_export.xml"
    with open(file_path, 'r', encoding='UTF-8') as infile, open(temp_file_path, 'w', encoding='UTF-8') as outfile:
        skip_dtd = False
        for line in infile:
            if '<!DOCTYPE' in line:
                skip_dtd = True
            if not skip_dtd:
                line = strip_invisible_character(line)
                outfile.write(line)
            if ']>' in line:
                skip_dtd = False

    print("done!")
    return temp_file_path


def xml_to_csv(file_path):
    """Loops through the element tree, retrieving all objects, and then
    combining them together into a dataframe
    """

    print("Converting XML File to CSV...", end="")
    sys.stdout.flush()

    attribute_list = []

    for event, elem in ET.iterparse(file_path, events=('end',)):
        if event == 'end':
            child_attrib = elem.attrib
            for metadata_entry in list(elem):
                metadata_values = list(metadata_entry.attrib.values())
                if len(metadata_values) == 2:
                    metadata_dict = {metadata_values[0]: metadata_values[1]}
                    child_attrib.update(metadata_dict)
            attribute_list.append(child_attrib)

            # Clear the element from memory to avoid excessive memory consumption
            elem.clear()

    health_df = pd.DataFrame(attribute_list)

    # Every health data type and some columns have a long identifer
    # Removing these for readability
    health_df.type = health_df.type.str.replace('HKQuantityTypeIdentifier', "")
    health_df.type = health_df.type.str.replace('HKCategoryTypeIdentifier', "")
    health_df.columns = health_df.columns.str.replace("HKCharacteristicTypeIdentifier", "")

    # Reorder some of the columns for easier visual data review
    original_cols = list(health_df)
    shifted_cols = ['type',
                    'sourceName',
                    'value',
                    'unit',
                    'startDate',
                    'endDate',
                    'creationDate']

    # Add loop specific column ordering if metadata entries exist
    if 'com.loopkit.InsulinKit.MetadataKeyProgrammedTempBasalRate' in original_cols:
        shifted_cols.append(
            'com.loopkit.InsulinKit.MetadataKeyProgrammedTempBasalRate')

    if 'com.loopkit.InsulinKit.MetadataKeyScheduledBasalRate' in original_cols:
        shifted_cols.append(
            'com.loopkit.InsulinKit.MetadataKeyScheduledBasalRate')

    if 'com.loudnate.CarbKit.HKMetadataKey.AbsorptionTimeMinutes' in original_cols:
        shifted_cols.append(
            'com.loudnate.CarbKit.HKMetadataKey.AbsorptionTimeMinutes')

    remaining_cols = list(set(original_cols) - set(shifted_cols))
    reordered_cols = shifted_cols + remaining_cols
    health_df = health_df.reindex(labels=reordered_cols, axis='columns')

    # Sort by newest data first
    health_df.sort_values(by='startDate', ascending=False, inplace=True)

    print("done!")

    return health_df


def save_to_csv(health_df, save_path):
    print("Saving CSV file...", end="")
    sys.stdout.flush()
    
    health_df.to_csv(save_path, index=False)
    print("done!")

    return

def remove_temp_file(temp_file_path):
    print("Removing temporary file...", end="")
    os.remove(temp_file_path)
    print("done!")
    
    return

### Dynamically finding the path the the apple folder regardless of what CWD context we're running in (either in notebook or orchestrator file) 

In [ ]:
## Initializing the main repo as none (we're looking for the main repo)
repo_path = None

for x in range(5):
    # Use .parents to go up a level in the file path, x is a number list how far back we want to go 0 = parent, 1 = grandparent, and so on
    candidate_path = Path.cwd().parents[x]

    # For each candidate path we're checking if main.py exists because that indicates we're at the main repo
    if (candidate_path / "main.py").is_file():
        repo_path = candidate_path
        print(f"Working directory found: '{repo_path}' ")
        break  # exits the outer loop too

## Raising an error here if we can't find apple folder using main.py indicator
if repo_path is None:
    raise FileNotFoundError("Could not find main.py")

## Building the apple folder file path now that we've found the main repo 
apple_folder_path = repo_path / "apple"

Working directory found: 'c:\Users\ryani\Coding Projects\habit_tracker_website' 
Found Raw Apple XML file


### Dynamically finding the raw apple xml file we're placed

In [ ]:
## Initializing the raw apple xml file path (this is what we're searching the apple folder for)
apple_raw_xml = None

for file in os.listdir(fr"{apple_folder_path}"):
    if file == "export.xml":
        apple_raw_xml = file
        print(f"Found Raw Apple XML file")
        break  # stop once we find it

## Exiting the script if we're unable to find the raw apple data xml file within the apple folder path
if apple_raw_xml is None:
    print(f"No Raw Apple XML file found... Exiting script")
    time.sleep(5)
    sys.exit(1)

# Building the path to the raw apple xml file we need to read in and transform
apple_raw_xml_path = Path(apple_folder_path, apple_raw_xml)

# Building the to save the output of our new apple csv file after it's been preprocessed 
save_path = Path(apple_folder_path, f"apple_health_export_{today}.csv" )

### Applying xml to csv functions 

In [6]:
# Creating a temporary xml file in the current repo
temp_file_path = preprocess_to_temp_file(apple_raw_xml_path)

# Reading the temporary xml file to clean and create a new csv file
health_df = xml_to_csv(temp_file_path)

# Saving the newly processed csv file to the apple folder 
save_to_csv(health_df, save_path)

# Removing the temporary xml file we initially created 
remove_temp_file(temp_file_path)

Pre-processing and writing to temporary file...done!
Converting XML File to CSV...done!
Saving CSV file...done!
Removing temporary file...done!


# Updating apple_raw table

## Grabbing the existing max date

In [8]:
# Grabbing the max date from the sqlite db so we only import new data
# When there is no existing data to grab a date from we're setting max_date as None
from pandas.errors import DatabaseError

try:
    with engine.connect() as connection:
        max_date_db = pd.read_sql_query(
            text("""SELECT MAX(DATETIME(adr.startDate)) as 'Max_Raw_UTC_Start_Date'
                    FROM apple_data_raw adr"""), 
            connection,
            parse_dates=['Max_Raw_UTC_Start_Date'])   

    max_date_db["Max_Raw_UTC_Start_Date"] = (
        pd.to_datetime(max_date_db["Max_Raw_UTC_Start_Date"])
        .dt.tz_localize("UTC"))

    max_UTC_date_db = max_date_db["Max_Raw_UTC_Start_Date"].iloc[0]

# When we can't find previous data to pull (fresh install) max date is set to None
except DatabaseError:
    max_UTC_date_db = None

## ETL step

In [9]:
# ETL step of gathering new data from apple health and upploading it almost as it is apple_data_raw

# Using 'save_path' defined above to initally save the csv file to read it back in, low_memory=False to avoid dtype warnings
apple = pd.read_csv(save_path, low_memory=False)

# Fitler the source to only my Apple Watch (Could pull in other devices later like iPhone or gamin)
apple = apple[(apple['sourceName'] == "Ryan’s Apple\xa0Watch") | (apple['sourceName'].isna())]

# Strip the "HKWorkoutActivityType" prefix from the workout types
apple["workoutActivityType"] = apple["workoutActivityType"].str.replace("HKWorkoutActivityType", "")

# Drop the UUID column, apple added this since last 7/18 upload
apple.drop(columns=['uuid'], inplace=True)

# Creating a copy of apple dataframe to avoid settingwithcopy warnings when converting datetime
apple_copied = apple.copy()

# Convert date columns to datetime format including UTC timezone
datetime_cols = ['startDate', 'endDate', 'creationDate']
for col in datetime_cols:
    apple_copied[col] = pd.to_datetime(apple_copied[col], format='%Y-%m-%d %H:%M:%S %z', utc=True)

# Using the exisiting max date in my db to filter to only new data from apple 
if max_UTC_date_db is not None:
    apple_copied = apple_copied[apple_copied["startDate"] > max_UTC_date_db] # dt_utc_test  max_UTC_date_db

# Cleaning the env
del apple

print("Apple raw ETL process completed, ready to appened to apple_raw table")

Apple raw ETL process completed, ready to appened to apple_raw table


## Appending new data to apple_raw table

In [10]:
# Appending the new raw data to the apple_data_raw table
apple_copied.to_sql(name="apple_data_raw", con=engine, if_exists='append', index=False)

print("new apple_raw data appened to table")

new apple_raw data appened to table


# Cleaning apple_raw to apple_workouts 

In [10]:

# Reading in only the newly updated data from apple_data_raw (using inital max_UTC_date_db prior to uploading new raw data)
with engine.connect() as connection: 
    apple_raw = pd.read_sql_query( 
        text("""
            SELECT workoutActivityType, startDate, type, maximum, minimum, sum, average, duration, durationUnit 
            FROM apple_data_raw
            WHERE startDate > :max_date
            """), connection,   # Kinda odd looks like max_UTC_date_db gets carried over into apple_raw (shouldn't matter though
                                # because we're only fitering to actual workouts)
        params={"max_date": str(max_UTC_date_db)}, # Using the str version of the datetime for filtering
        parse_dates=['startDate']) # Parse date here takes out need for pd.to_datetime below 

apple_raw["startDate"] = apple_raw["startDate"].dt.tz_localize("UTC")

# Filters df to include only rows where at least one of the workout-related columns is not missing (NaN).
# .notna() Check for Non-Nulls in selected columns 
# .any(axis=1) aggregates rows where there's at least one non-missing value 
apple_workouts = apple_raw[apple_raw.notna().any(axis=1)] 

print("New apple workout data successfully read in from the apple raw table")

New apple workout data successfully read in from the apple raw table


In [11]:
df_long = pd.melt(apple_workouts, 
                    id_vars=['startDate', 'workoutActivityType', 'type', 'duration', 'durationUnit'],
                    value_vars=['maximum', 'minimum', 'sum', 'average'], # turned into rows via where the name goes into 'measurement_type' and actual values to the 'value' column
                    var_name='measurement_type', 
                    value_name='value')

# Removing rows where either type or value are null (no meaningful data here)
df_long = df_long[(df_long['type'].notna()) | (df_long['value'].notna())]

# Handle duration rows separately and combine
duration_rows = apple_workouts[apple_workouts['duration'].notna() & apple_workouts['type'].isna()].copy()
duration_rows = duration_rows.assign(
    measurement_type='total',
    value=duration_rows['duration'],
    type='Duration')[['startDate', 'workoutActivityType', 'type', 'measurement_type', 'value', 'durationUnit']]

# Combine and clean
aw_long = pd.concat([df_long[df_long['value'].notna()][['startDate', 'workoutActivityType', 'type', 'measurement_type', 'value', 'durationUnit']],
    duration_rows], ignore_index=True)

# Rename columns for clarity
aw_long.rename(columns={'startDate': 'StartDate', # Why do I rename StartDate to be different than apple data raw? 
                        'workoutActivityType': 'activity', 
                        'type': 'metric', 
                        'measurement_type': 'measurement_type', 
                        'value': 'value', 
                        'durationUnit': 'd_unit'}, inplace=True)

In [12]:
# Reshape the apple_workouts DataFrame from wide format (one row per workout with multiple measurement columns) to long format (one row per measurement per workout).
df_long = pd.melt(apple_workouts, 
                    id_vars=['startDate', 'workoutActivityType', 'type', 'duration', 'durationUnit'],
                    value_vars=['maximum', 'minimum', 'sum', 'average'], # turned into rows via where the name goes into 'measurement_type' and actual values to the 'value' column
                    var_name='measurement_type', 
                    value_name='value')

# Removing rows where either type or value are null (no meaningful data here)
df_long = df_long[(df_long['type'].notna()) | (df_long['value'].notna())]

# Handle duration rows separately and combine
duration_rows = apple_workouts[apple_workouts['duration'].notna() & apple_workouts['type'].isna()].copy()
duration_rows = duration_rows.assign(
    measurement_type='total',
    value=duration_rows['duration'],
    type='Duration')[['startDate', 'workoutActivityType', 'type', 'measurement_type', 'value', 'durationUnit']]

# Combine and clean
aw_long = pd.concat([df_long[df_long['value'].notna()][['startDate', 'workoutActivityType', 'type', 'measurement_type', 'value', 'durationUnit']],
    duration_rows], ignore_index=True)

# Rename columns for clarity
aw_long.rename(columns={'startDate': 'StartDate', 'workoutActivityType': 'activity', 'type': 'metric', # Why do I rename StartDate to be different than apple data raw?
                       'measurement_type': 'measurement_type', 'value': 'value', 'durationUnit': 'd_unit'}, inplace=True)

In [13]:
# Defining the function to grab current max workout_id and count from there
def max_workout_id():
    # Checking to see if table is active

    #Gathering the max workout_id from existing apple_workouts table
    with engine.connect() as connection:
        max_workout_id_df = pd.read_sql_query(
                text("""SELECT max(workout_id) as workout_id 
                        FROM apple_workouts
                        WHERE activity IS NOT NULL"""), # Need to filter on activity because otherwise workout_id is NA
                connection)  
    # If Apple workouts is not created yet, this type block will catch it and automatically set workout_id to 0 
    try:
        # Grabbing singlular max workout_id from dataset to filter new data off of  
        max_workout_id = int(max_workout_id_df['workout_id'][0]) 

    # If the table is empty, set max_workout_id to 0
    except TypeError:
        max_workout_id = 0 

    return max_workout_id

In [14]:
if not aw_long.empty:

    # cleaning value column to only go out 2 decimal places
    aw_long['value'] = aw_long['value'].round(2)

    aw_long.sort_values('StartDate', ascending=False, inplace=True)

    # We're able to easily get unique workouts by date because aw_long only has activity filled for the duration metric which every workout has 1 of 
    activity_map = aw_long[aw_long['activity'].notnull()][["StartDate", "activity"]]

    # Sorting values by date and resetting index (easy way to grab a unique id so long as I'm sorting correctly)
    activity_map = activity_map.sort_values('StartDate').reset_index(drop=True)

    # Creating a new variable based on the index  
    # Previous null values probably from Garmin are keeping workout_id from being an integer in aw_final
    activity_map['workout_id'] = activity_map.index.astype(int) + 1 + max_workout_id()  # Start IDs from 1

    # Merge activity back onto the main dataframe using startDate
    aw_final = pd.merge(aw_long, activity_map, on='StartDate', how='left', suffixes=('', '_Specifier'))

    # Drop the original activity column
    aw_final.drop(columns=['activity'], inplace=True)

    # Rename the helper column to activity since that's actually what we'll want
    aw_final.rename(columns={'activity_Specifier': 'activity'}, inplace=True)

    # Adding new activity type variable for minutes calculation 
    aw_final['activity_type'] = np.where(aw_final['activity'].isin(['Running', 'Cycling', 'Swimming', 'Walking']), 'Cardio',
        np.where(aw_final['activity'] == 'TraditionalStrengthTraining', 'Weights', None))
    
    if max_UTC_date_db is not None:
        print(f"Uploaded apple data from {str(max_UTC_date_db)} to {str(max(apple_workouts['startDate']))} ")

else:
    print("No new apple workouts to upload today \n")

No new apple workouts to upload today 



## Uploading new date to apple_workout table

In [16]:
#def final_upload(aw_final):
aw_final.to_sql(name="apple_workouts", con=engine, if_exists='append', index=False)

print("New apple workout data appended to apple_workouts table")

New apple workout data appended to apple_workouts table
